# Transformer 交互式学习 Notebook

这个 Notebook 带你逐步理解和实验 Transformer 模型的各个组件。

## 目录
1. [环境准备](#环境准备)
2. [词嵌入与位置编码](#词嵌入与位置编码)
3. [自注意力机制](#自注意力机制)
4. [多头注意力](#多头注意力)
5. [Encoder-Decoder 结构](#Encoder-Decoder 结构)
6. [完整模型实验](#完整模型实验)

In [ ]:
# 环境准备
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

# 导入我们实现的 Transformer 组件
from transformer_model import (
    PositionalEncoding, MultiHeadAttention, FeedForward, LayerNorm,
    EncoderLayer, DecoderLayer, Encoder, Decoder, Transformer,
    SMALL_CONFIG
)

# 设置随机种子保证可复现性
torch.manual_seed(42)
np.random.seed(42)

# 设置图表显示
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 词嵌入与位置编码

In [ ]:
# 1. 词嵌入演示
vocab_size = 100
d_model = 64

embedding = nn.Embedding(vocab_size, d_model)

# 创建一个简单的输入序列
input_ids = torch.tensor([[10, 25, 30, 15, 50]])
embedded = embedding(input_ids)

print(f"输入形状：{input_ids.shape}")
print(f"嵌入形状：{embedded.shape}")
print(f"\n词嵌入将离散的 token ID 映射到连续的向量空间")

In [ ]:
# 2. 位置编码可视化
pe = PositionalEncoding(d_model=d_model, max_len=50)

# 可视化位置编码
pe_matrix = pe.pe[0, :50, :].numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 热力图
sns.heatmap(pe_matrix, ax=axes[0], cmap='coolwarm', center=0)
axes[0].set_title('位置编码矩阵 (全部维度)')
axes[0].set_xlabel('维度')
axes[0].set_ylabel('位置')

# 曲线图 - 前几个维度
positions = np.arange(50)
for i in range(0, 8, 2):
    axes[1].plot(positions, pe_matrix[:, i], label=f'维度 {i}')
axes[1].set_title('不同维度的位置编码曲线')
axes[1].set_xlabel('位置')
axes[1].set_ylabel('值')
axes[1].legend()

plt.tight_layout()
plt.show()

print("位置编码使用不同频率的正弦/余弦函数，使模型能够学习相对位置关系")

## 自注意力机制

In [ ]:
# 单头注意力演示
d_model = 64
num_heads = 1
seq_len = 10

attn = MultiHeadAttention(d_model, num_heads)

# 随机输入序列
x = torch.randn(1, seq_len, d_model)

# 自注意力：Q=K=V=x
output, attn_weights = attn(x, x, x)

print(f"输入形状：{x.shape}")
print(f"输出形状：{output.shape}")
print(f"注意力权重形状：{attn_weights.shape}")

# 可视化注意力矩阵
plt.figure(figsize=(8, 6))
sns.heatmap(attn_weights[0, 0].detach().numpy(), annot=True, fmt='.2f', cmap='Blues')
plt.title('自注意力权重矩阵')
plt.xlabel('Key 位置')
plt.ylabel('Query 位置')
plt.tight_layout()
plt.show()

In [ ]:
# 多头注意力演示
num_heads = 4
attn_multi = MultiHeadAttention(d_model, num_heads)

output_multi, weights_multi = attn_multi(x, x, x)

print(f"多头注意力头数：{num_heads}")
print(f"每头维度：{d_model // num_heads}")

# 可视化多个头的注意力
fig, axes = plt.subplots(1, num_heads, figsize=(4*num_heads, 3))
for h in range(num_heads):
    sns.heatmap(weights_multi[0, h].detach().numpy(), ax=axes[h], cmap='Blues', 
                vmin=0, vmax=1, annot=True, fmt='.2f')
    axes[h].set_title(f'Head {h}')
    axes[h].set_xlabel('Key')
    if h == 0:
        axes[h].set_ylabel('Query')
    else:
        axes[h].set_yticks([])

plt.tight_layout()
plt.show()

print("\n多头注意力允许模型同时关注不同的表示子空间")

## Encoder-Decoder 结构

In [ ]:
# 创建完整的 Encoder 和 Decoder
config = SMALL_CONFIG
vocab_size = 50

encoder = Encoder(vocab_size, **config)
decoder = Decoder(vocab_size, **config)

print(f"Encoder 参数量：{sum(p.numel() for p in encoder.parameters()):,}")
print(f"Decoder 参数量：{sum(p.numel() for p in decoder.parameters()):,}")

In [ ]:
# 演示 Encoder-Decoder 数据流
batch_size = 2
src_len = 8
tgt_len = 6

src = torch.randint(1, vocab_size, (batch_size, src_len))
tgt = torch.randint(1, vocab_size, (batch_size, tgt_len))

print("=== Encoder-Decoder 数据流 ===")
print(f"\n输入:")
print(f"  源序列 (src): {src.shape}")
print(f"  目标序列 (tgt): {tgt.shape}")

# Encoder 前向传播
enc_output, enc_attn = encoder(src)
print(f"\nEncoder 输出：{enc_output.shape}")

# Decoder 前向传播
dec_output, dec_self_attn, dec_cross_attn = decoder(tgt, enc_output)
print(f"Decoder 输出：{dec_output.shape}")

print(f"\n注意力权重:")
print(f"  Encoder: {len(enc_attn)} 层，每层形状 {enc_attn[0].shape}")
print(f"  Decoder Self-Attn: {len(dec_self_attn)} 层")
print(f"  Decoder Cross-Attn: {len(dec_cross_attn)} 层")

## 完整模型实验

In [ ]:
# 创建完整 Transformer 模型
model = Transformer(vocab_size, **config)

print(f"完整 Transformer 参数量：{sum(p.numel() for p in model.parameters()):,}")
print(f"\n模型配置:")
for k, v in config.items():
    print(f"  {k}: {v}")

In [ ]:
# 前向传播测试
model.eval()
with torch.no_grad():
    output, attn_info = model(src, tgt)

print(f"输出形状：{output.shape}")
print(f"每个位置预测词表概率分布：{output.shape[-1]} (vocab_size)")

# 获取预测
predictions = output.argmax(dim=-1)
print(f"\n预测的 token IDs: {predictions[0].tolist()}")

In [ ]:
# 可视化完整模型的注意力
last_layer = -1

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Encoder 自注意力
enc_attn = attn_info['encoder_attn'][last_layer][0, 0].numpy()
sns.heatmap(enc_attn, ax=axes[0], cmap='Blues')
axes[0].set_title('Encoder 自注意力')
axes[0].set_xlabel('Key')
axes[0].set_ylabel('Query')

# Decoder 自注意力
dec_self = attn_info['decoder_self_attn'][last_layer][0, 0].numpy()
sns.heatmap(dec_self, ax=axes[1], cmap='Blues')
axes[1].set_title('Decoder 自注意力')
axes[1].set_xlabel('Key')
axes[1].set_ylabel('Query')

# Decoder 交叉注意力
dec_cross = attn_info['decoder_cross_attn'][last_layer][0].mean(0).numpy()
sns.heatmap(dec_cross[:tgt_len, :src_len], ax=axes[2], cmap='Blues')
axes[2].set_title('Decoder 交叉注意力')
axes[2].set_xlabel('Encoder 位置')
axes[2].set_ylabel('Decoder 位置')

plt.tight_layout()
plt.show()

## 简单训练实验

In [ ]:
# 创建一个简单的训练任务
def create_data(num_samples=200, seq_len=6, vocab_size=30):
    X = torch.randint(1, vocab_size-5, (num_samples, seq_len))
    y = torch.roll(X, shifts=-1, dims=1)  # 循环移位作为目标
    return X, y

X, y = create_data()
print(f"训练数据形状：X={X.shape}, y={y.shape}")
print(f"样本输入：{X[0].tolist()}")
print(f"样本输出：{y[0].tolist()}")

In [ ]:
# 训练模型
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20
losses = []

model.train()
for epoch in range(epochs):
    optimizer.zero_grad()
    
    # Decoder 输入是目标去掉最后一个 token
    tgt_input = y[:, :-1]
    # 真实标签是目标去掉第一个 token
    tgt_true = y[:, 1:]
    
    output, _ = model(X, tgt_input)
    loss = criterion(output.view(-1, output.size(-1)), tgt_true.view(-1))
    
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

# 绘制损失曲线
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('训练损失曲线')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 测试训练后的模型
model.eval()
test_input = X[:3]
test_target = y[:3]

with torch.no_grad():
    output, _ = model(test_input, test_target[:, :-1])
    pred = output.argmax(dim=-1)

print("=== 模型预测测试 ===")
for i in range(3):
    print(f"\n样本 {i+1}:")
    print(f"  输入：    {test_input[i].tolist()}")
    print(f"  期望输出：{test_target[i, 1:].tolist()}")
    print(f"  预测输出：{pred[i].tolist()}")
    accuracy = (pred[i] == test_target[i, 1:]).sum().item() / len(pred[i]) * 100
    print(f"  准确率：{accuracy:.1f}%")